# 03 · Configure mandatory & optional steps

The workflow has five **mandatory** steps and two **optional** ones (*axiom finding* and *human-in-the-loop evaluation*). This notebook shows the configuration helpers and the human-in-the-loop gate.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # make the package importable
import matplotlib
matplotlib.use('Agg')  # headless-safe; remove for interactive plots
import bottomup_ontology as bo
print('bottomup_ontology', bo.__version__)

bottomup_ontology 0.1.0


In [2]:
corpus = [
    'A rugby player plays in a position. Siya Kolisi is a rugby player. '
    'A team has many players.',
    'A club has a coach. The coach trains the team. A player belongs to a club.',
    'Mammals such as lions and impalas live in the savanna. A lion eats impala.',
    'A blog post has a title and comments. A blog post has a category.',
]
gold_classes = {'rugby player','player','team','club','coach','lion',
                'impala','position','blog post','category','mammal'}
len(corpus)

4

## Minimal vs. full pipeline
`build_minimal_workflow` keeps only mandatory steps; `build_default_workflow` enables everything.

In [3]:
print('minimal:', bo.execution_plan(bo.build_minimal_workflow()))
print('default:', bo.execution_plan(bo.build_default_workflow()))

minimal: ['text_cleaning', 'preprocessing', 'term_extraction', 'relation_extraction', 'evaluation']
default: ['text_cleaning', 'preprocessing', 'term_extraction', 'relation_extraction', 'axiom_finding', 'human_in_the_loop', 'evaluation']


## Toggle optional steps declaratively with `configure_workflow`

In [4]:
g = bo.configure_workflow(include_axiom_finding=True,
                          include_human_in_the_loop=False)
print('axiom only :', bo.execution_plan(g))
g = bo.configure_workflow(include_axiom_finding=False,
                          include_human_in_the_loop=True)
print('HITL only  :', bo.execution_plan(g))

axiom only : ['text_cleaning', 'preprocessing', 'term_extraction', 'relation_extraction', 'axiom_finding', 'evaluation']
HITL only  : ['text_cleaning', 'preprocessing', 'term_extraction', 'relation_extraction', 'human_in_the_loop', 'evaluation']


## Add / remove optional steps on an existing graph
These return a **new** graph re-wired around the change (mandatory steps cannot be removed).

In [5]:
g = bo.build_minimal_workflow()
print('start          :', bo.execution_plan(g))
g = bo.add_optional_step(g, 'axiom_finding')
print('+axiom_finding :', bo.execution_plan(g))
g = bo.add_optional_step(g, 'human_in_the_loop')
print('+HITL          :', bo.execution_plan(g))
g = bo.remove_optional_step(g, 'axiom_finding')
print('-axiom_finding :', bo.execution_plan(g))
try:
    bo.remove_optional_step(g, 'term_extraction')
except ValueError as e:
    print('cannot remove mandatory:', e)

start          : ['text_cleaning', 'preprocessing', 'term_extraction', 'relation_extraction', 'evaluation']
+axiom_finding : ['text_cleaning', 'preprocessing', 'term_extraction', 'relation_extraction', 'axiom_finding', 'evaluation']
+HITL          : ['text_cleaning', 'preprocessing', 'term_extraction', 'relation_extraction', 'axiom_finding', 'human_in_the_loop', 'evaluation']
-axiom_finding : ['text_cleaning', 'preprocessing', 'term_extraction', 'relation_extraction', 'human_in_the_loop', 'evaluation']
cannot remove mandatory: cannot remove mandatory step 'term_extraction'


## Per-step parameters
`term_extraction` exposes `min_frequency` and `top_k`. Pass defaults at build time via `step_params`, or per-run via `run_workflow(..., step_params=...)`.

In [6]:
g = bo.configure_workflow(step_params={'term_extraction': {'top_k': 6}})
st = bo.run_workflow(g, bo.WorkflowState(documents=corpus))
print('top_k=6 -> classes:', sorted(st.ontology.classes))

top_k=6 -> classes: ['blog post', 'club', 'coach', 'lion', 'live savanna', 'mammal', 'player', 'rugby player', 'siya kolisi', 'team']


## The human-in-the-loop gate

Naive extraction produces some noise — e.g. the Hearst pattern over *"Mammals such as lions and impalas live in the savanna"* yields a spurious class `live savanna`. The optional **human-in-the-loop** step uses an approver callback `(kind, label) -> bool` to prune candidates before final evaluation.

In [7]:
# Baseline: run WITHOUT the human-in-the-loop step
g_no = bo.configure_workflow(include_human_in_the_loop=False)
st_no = bo.run_workflow(g_no, bo.WorkflowState(documents=corpus,
                                               gold_classes=gold_classes))
print('without HITL -> classes:', len(st_no.ontology.classes))
print('  ', sorted(st_no.ontology.classes))
print('  class P/R/F1:', st_no.evaluation['class_prf'])

without HITL -> classes: 18
   ['blog post', 'category', 'club', 'coach', 'comment', 'impala', 'kolisi', 'lion', 'live savanna', 'mammal', 'player', 'position', 'post', 'rugby player', 'savanna', 'siya kolisi', 'team', 'title']
  class P/R/F1: {'precision': 0.611, 'recall': 1.0, 'f1': 0.759, 'tp': 11, 'support': 11}


In [8]:
# A domain expert standing in as a callback. Here we reject obvious
# noise: multi-word 'classes' whose head is a verb-like token, and any
# class not anchored in the domain vocabulary.
NOISE = {'live savanna', 'kolisi'}

def approver(kind, label):
    label = label.lower()
    if kind == 'class':
        return label not in NOISE
    return True  # accept all object properties

g_yes = bo.configure_workflow(include_human_in_the_loop=True)
st_yes = bo.run_workflow(
    g_yes,
    bo.WorkflowState(documents=corpus, gold_classes=gold_classes,
                     human_approver=approver),
)
print('with HITL -> classes:', len(st_yes.ontology.classes))
print('  ', sorted(st_yes.ontology.classes))
print('  class P/R/F1:', st_yes.evaluation['class_prf'])
print()
removed = set(st_no.ontology.classes) - set(st_yes.ontology.classes)
print('pruned by the expert:', sorted(removed))

with HITL -> classes: 16
   ['blog post', 'category', 'club', 'coach', 'comment', 'impala', 'lion', 'mammal', 'player', 'position', 'post', 'rugby player', 'savanna', 'siya kolisi', 'team', 'title']
  class P/R/F1: {'precision': 0.688, 'recall': 1.0, 'f1': 0.815, 'tp': 11, 'support': 11}

pruned by the expert: ['kolisi', 'live savanna']
